# Documentation de la classe `Hydrology`

La classe `Hydrology` (`kadi.weather.hydrology.Hydrology`) gère la modélisation hydrologique pour une parcelle au Bénin. Elle permet de simuler le bilan hydrique du sol en appliquant les recommandations de la méthode **FAO-56**.

## 1. Initialisation de la Modélisation
Pour instancier l'analyseur hydrologique, vous devez fournir :
- Une instance `Location`
- Une série de précipitations (`pandas.Series`)
- Un `DataFrame` de températures (`temperature_min`, `temperature_max`)
- Le type de sol (si omis, le système tente de le résoudre via le cache de `SoilGrids`)
- La culture étudiée (`crop`)

In [1]:
# Changer de répertoire
import os
from pathlib import Path
print(Path.cwd())
new_dir = Path("~/Bureau/kadipy/").expanduser().resolve()
os.chdir(new_dir)
print(Path.cwd())

from kadi.weather.location import Location
from kadi.weather.data import WeatherData
from kadi.weather.hydrology import Hydrology

# On récupère d'abord les données météo de base
loc = Location(latitude=9.3333, longitude=2.6333, name='Parakou')
weather = WeatherData(loc)
hist = weather.fetch_historical(months_back=1)

# Initialisation de l'analyseur hydrologique
hydro = Hydrology(loc, hist['precipitation'], hist[['temperature_min', 'temperature_max']])

print(f"Culture étudiée : {hydro.crop}")
print(f"Type de sol identifié : {hydro.soil_type}")
print("Paramètres du sol :", hydro.soil_params)

/home/dels/Bureau/kadipy/docs/weather
/home/dels/Bureau/kadipy
Culture étudiée : maize
Type de sol identifié : ferrugineux
Paramètres du sol : {'taw': 100.0, 'cn_amc2': 82.0, 'ksat': 15.0}


## 2. Modélisation de l'Évapotranspiration (ET0)
**Méthode :** `et0_hargreaves(tmin: float, tmax: float, day_of_year: int) -> float`

KadiPy utilise la formule empirique de **Hargreaves-Samani** pour calculer l'évapotranspiration de référence (ET0). Cette méthode est particulièrement adaptée à l'Afrique de l'Ouest car elle nécessite uniquement les températures min/max et la latitude (pour déduire le rayonnement solaire extraterrestre), palliant ainsi le manque de données sur la vitesse du vent ou l'humidité relative exacte.

In [2]:
# Exemple d'évapotranspiration au 150ème jour de l'année (fin mai)
et0 = hydro.et0_hargreaves(tmin=23.0, tmax=35.0, day_of_year=150)
print(f"ET0 estimée (Hargreaves-Samani) : {et0:.2f} mm/jour")

ET0 estimée (Hargreaves-Samani) : 5.63 mm/jour


## 3. Modélisation du Ruissellement (Méthode SCS-CN)
**Méthode :** `runoff_cn(precipitation: float, prior_5d_rain: float = 0.0) -> float`

KadiPy modélise le ruissellement de surface en utilisant la méthode révisée **SCS-CN** (Soil Conservation Service - Curve Number). Le paramètre `Curve Number` dépend du type de sol. 
Le modèle intègre également un ajustement **AMC** (Antecedent Moisture Condition) : l'état d'humidité du sol sur les 5 jours précédents fait varier la courbe de ruissellement (sols secs vs sols saturés).

In [3]:
# Sur un sol ferrugineux après 5 jours sans pluie (AMC I) avec une forte averse (40mm)
runoff_dry = hydro.runoff_cn(precipitation=40.0, prior_5d_rain=0.0)

# Sur le même sol après 5 jours très pluvieux (AMC III - sol saturé)
runoff_wet = hydro.runoff_cn(precipitation=40.0, prior_5d_rain=50.0)

print(f"Ruissellement (Sol Sec) : {runoff_dry:.2f} mm")
print(f"Ruissellement (Sol Saturé) : {runoff_wet:.2f} mm")

Ruissellement (Sol Sec) : 1.48 mm
Ruissellement (Sol Saturé) : 21.03 mm


## 4. Calcul du Bilan Hydrique Global
**Méthode :** `compute_water_balance() -> pd.DataFrame`

C'est le moteur principal qui boucle sur l'historique journalier. Il intègre le type de sol, le ruissellement (SCS-CN), et l'évapotranspiration (Hargreaves + Coefficients culturaux `Kc`) pour évaluer le déficit quotidien et la réserve utile (TAW - Total Available Water).
La série temporelle générée aide à évaluer l'indice de stress hydrique.

In [4]:
# Paramétrage forcé pour la démonstration
hydro.crop = 'maize'
hydro.soil_type = 'ferrugineux'
hydro.soil_params = hydro.get_soil_params('ferrugineux')

# Exécution de la simulation sur 1 mois
bilan_df = hydro.compute_water_balance()

display(bilan_df.tail())

,precip,et0,pluie_eff,evapotransp,deficit_eau,reserve_utile,stress_hydrique_index
date,,,,,,,
2026-06-26,1.3,4.64,1.3,5.56,41.05,58.95,0.41
2026-06-27,0.6,4.09,0.6,4.91,45.36,54.64,0.45
2026-06-28,1.4,4.51,1.4,5.42,49.38,50.62,0.49
2026-06-29,20.3,3.80,20.3,4.56,33.63,66.37,0.34
2026-06-30,8.6,3.99,8.6,4.78,29.82,70.18,0.30


## 5. Dictionnaires Internes (Sol & Cultures)
La classe contient deux méthodes qui fournissent les paramètres agronomiques de base :
- `get_soil_params(soil_type: str)` : Fournit la réserve totale (TAW), le Curve Number (`cn_amc2`) et la conductivité (`ksat`) pour des sols locaux (ferrugineux, ferrallitique, sableux, limoneux).
- `get_crop_coefficients(crop: str, stage: str)` : Fournit les coefficients culturaux (`Kc`) de la FAO (pour l'instant la valeur moyenne `mid` est utilisée dans le MVP) pour le maïs, riz, sorgho, manioc et tomates.